## Example 1: asgi_lifespan functionality test 1

In [ ]:
import asgi_lifespan
from asgi_lifespan import LifespanManager

# Example 1: Basic ASGI lifespan management
print("Example 1: Testing basic ASGI lifespan functionality")

try:
    print("asgi_lifespan module imported successfully")
    print("Provides lifespan management for ASGI applications")
    
    # Check available components
    attrs = [attr for attr in dir(asgi_lifespan) if not attr.startswith('_')]
    print(f"Available components: {attrs}")
    
    # Basic ASGI application for testing
    async def sample_asgi_app(scope, receive, send):
        if scope["type"] == "lifespan":
            message = await receive()
            if message["type"] == "lifespan.startup":
                print("Application startup event")
                await send({"type": "lifespan.startup.complete"})
            elif message["type"] == "lifespan.shutdown":
                print("Application shutdown event")
                await send({"type": "lifespan.shutdown.complete"})
        else:
            # Handle other request types
            await send({
                "type": "http.response.start",
                "status": 200,
                "headers": []
            })
            await send({
                "type": "http.response.body",
                "body": b"Hello World"
            })
    
    print("Sample ASGI application created")
    print("ASGI lifespan management setup ready")
    
    # Test LifespanManager creation
    manager = LifespanManager(sample_asgi_app)
    print(f"LifespanManager created: {type(manager).__name__}")
    
    print("asgi_lifespan example 1 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"asgi_lifespan example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: asgi_lifespan functionality test 2

In [ ]:
import asgi_lifespan
from asgi_lifespan import LifespanManager
import asyncio

# Example 2: ASGI lifespan context manager usage
print("Example 2: Testing ASGI lifespan context manager")

try:
    # Create a more comprehensive ASGI app with lifespan events
    class AppState:
        def __init__(self):
            self.startup_completed = False
            self.shutdown_completed = False
    
    app_state = AppState()
    
    async def lifespan_asgi_app(scope, receive, send):
        if scope["type"] == "lifespan":
            # ASGI lifespan: keep receiving until shutdown (one receive() then return left LifespanManager waiting 5 s -> TimeoutError)
            while True:
                message = await receive()
                if message["type"] == "lifespan.startup":
                    print("Executing startup tasks...")
                    # Simulate startup tasks (database connection, cache initialization, etc.)
                    app_state.startup_completed = True
                    await send({"type": "lifespan.startup.complete"})
                elif message["type"] == "lifespan.shutdown":
                    print("Executing shutdown tasks...")
                    # Simulate cleanup tasks
                    app_state.shutdown_completed = True
                    await send({"type": "lifespan.shutdown.complete"})
                    return
    
    # Test lifespan manager as context manager
    print("Testing LifespanManager as context manager")
    
    async def test_lifespan_context():
        async with LifespanManager(lifespan_asgi_app) as manager:
            print("Inside lifespan context - app is running")
            print(f"Startup completed: {app_state.startup_completed}")
            assert app_state.startup_completed and not app_state.shutdown_completed
            
            # Simulate some work while app is running
            await asyncio.sleep(0.01)
            print("Simulated work completed")
        
        print("Exited lifespan context - app is shut down")
        print(f"Shutdown completed: {app_state.shutdown_completed}")
        assert app_state.shutdown_completed
    
    # Run the test (in Jupyter this works directly with await)
    await test_lifespan_context()
    
    print("Lifespan context manager working correctly")
    print("asgi_lifespan example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    import traceback
    traceback.print_exc()
    raise
    
print("Example 2 finished")

## Example 3: asgi_lifespan functionality test 3

In [ ]:
import asgi_lifespan
from asgi_lifespan import LifespanManager
import asyncio

# Example 3: Error handling in lifespan events
print("Example 3: Testing error handling in lifespan events")

try:
    # Create ASGI app that handles startup/shutdown errors
    class AppState:
        def __init__(self):
            self.database_connected = False
            self.cache_initialized = False
            self.startup_error = None
            self.shutdown_error = None
    
    app_state = AppState()
    
    async def error_handling_app(scope, receive, send):
        if scope["type"] == "lifespan":
            # ASGI lifespan: keep receiving until shutdown (one receive() then return left LifespanManager waiting 5 s -> TimeoutError)
            while True:
                message = await receive()
            
                if message["type"] == "lifespan.startup":
                    try:
                        print("Starting up application...")
                    
                        # Simulate database connection
                        await asyncio.sleep(0.01)
                        app_state.database_connected = True
                        print("Database connected")
                    
                        # Simulate cache initialization
                        await asyncio.sleep(0.01)
                        app_state.cache_initialized = True
                        print("Cache initialized")
                    
                        await send({"type": "lifespan.startup.complete"})
                    
                    except Exception as e:
                        app_state.startup_error = str(e)
                        print(f"Startup error: {e}")
                        await send({"type": "lifespan.startup.failed", "message": str(e)})
                        return
            
                elif message["type"] == "lifespan.shutdown":
                    try:
                        print("Shutting down application...")
                    
                        # Simulate cleanup tasks
                        if app_state.cache_initialized:
                            await asyncio.sleep(0.01)
                            app_state.cache_initialized = False
                            print("Cache cleared")
                    
                        if app_state.database_connected:
                            await asyncio.sleep(0.01)
                            app_state.database_connected = False
                            print("Database disconnected")
                    
                        await send({"type": "lifespan.shutdown.complete"})
                        return
                    
                    except Exception as e:
                        app_state.shutdown_error = str(e)
                        print(f"Shutdown error: {e}")
                        await send({"type": "lifespan.shutdown.failed", "message": str(e)})
                        return
    
    # Test successful startup and shutdown
    print("Testing successful lifespan management:")
    
    async def test_error_handling():
        async with LifespanManager(error_handling_app):
            print("Application is running")
            print(f"Database connected: {app_state.database_connected}")
            print(f"Cache initialized: {app_state.cache_initialized}")
            assert app_state.database_connected and app_state.cache_initialized
            await asyncio.sleep(0.01)
        
        print("Application shutdown completed")
        print(f"Final state - DB: {app_state.database_connected}, Cache: {app_state.cache_initialized}")
        assert not app_state.database_connected and not app_state.cache_initialized
        assert app_state.startup_error is None and app_state.shutdown_error is None
    
    await test_error_handling()
    
    print("Error handling in lifespan events working")
    print("asgi_lifespan example 3 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    import traceback
    traceback.print_exc()
    raise
    
print("Example 3 finished")

## Example 4: asgi_lifespan functionality test 4

In [ ]:
import asgi_lifespan
from asgi_lifespan import LifespanManager
import asyncio

# Example 4: Integration with ASGI frameworks
print("Example 4: Testing integration with ASGI frameworks")

try:
    # Simulate a FastAPI-like ASGI application structure
    class MockFrameworkApp:
        def __init__(self):
            self.startup_handlers = []
            self.shutdown_handlers = []
            self.state = {"db_pool": None, "cache": None}
        
        def add_event_handler(self, event_type, handler):
            if event_type == "startup":
                self.startup_handlers.append(handler)
            elif event_type == "shutdown":
                self.shutdown_handlers.append(handler)
        
        async def __call__(self, scope, receive, send):
            if scope["type"] == "lifespan":
                # ASGI lifespan: keep receiving until shutdown (one receive() then return left LifespanManager waiting 5 s -> TimeoutError)
                while True:
                    message = await receive()
                
                    if message["type"] == "lifespan.startup":
                        print("Framework startup sequence...")
                    
                        # Execute startup handlers
                        for handler in self.startup_handlers:
                            await handler()
                    
                        await send({"type": "lifespan.startup.complete"})
                
                    elif message["type"] == "lifespan.shutdown":
                        print("Framework shutdown sequence...")
                    
                        # Execute shutdown handlers in reverse order
                        for handler in reversed(self.shutdown_handlers):
                            await handler()
                    
                        await send({"type": "lifespan.shutdown.complete"})
                        return
            
            else:
                # Handle HTTP requests
                await send({
                    "type": "http.response.start",
                    "status": 200,
                    "headers": [(b"content-type", b"application/json")]
                })
                await send({
                    "type": "http.response.body", 
                    "body": b'{"message": "Framework app running"}'
                })
    
    # Create mock framework app
    app = MockFrameworkApp()
    
    # Add startup handlers
    async def initialize_database():
        print("Initializing database connection pool...")
        app.state["db_pool"] = "mock_db_pool"
        
    async def initialize_cache():
        print("Initializing Redis cache...")
        app.state["cache"] = "mock_cache_connection"
    
    # Add shutdown handlers
    async def cleanup_database():
        print("Closing database connections...")
        app.state["db_pool"] = None
        
    async def cleanup_cache():
        print("Closing cache connections...")
        app.state["cache"] = None
    
    # Register event handlers
    app.add_event_handler("startup", initialize_database)
    app.add_event_handler("startup", initialize_cache)
    app.add_event_handler("shutdown", cleanup_database)
    app.add_event_handler("shutdown", cleanup_cache)
    
    print("Framework app configured with startup/shutdown handlers")
    
    # Test with LifespanManager
    print("\nTesting framework integration:")
    
    async def test_framework_integration():
        async with LifespanManager(app):
            print("Framework app is running")
            print(f"App state: DB={app.state['db_pool']}, Cache={app.state['cache']}")
            assert app.state == {"db_pool": "mock_db_pool", "cache": "mock_cache_connection"}, app.state
            
            # Simulate some application work
            await asyncio.sleep(0.01)
            print("Framework handling requests...")
        
        print("Framework app shutdown completed")
        print(f"Final state: DB={app.state['db_pool']}, Cache={app.state['cache']}")
        assert app.state == {"db_pool": None, "cache": None}, app.state
    
    await test_framework_integration()
    
    print("Framework integration working correctly")
    print("asgi_lifespan example 4 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    import traceback
    traceback.print_exc()
    raise
    
print("Example 4 finished")

## Example 5: asgi_lifespan functionality test 5

In [ ]:
import asgi_lifespan
from asgi_lifespan import LifespanManager
import asyncio

# Example 5: asgi_lifespan package capabilities and testing utilities
print("Example 5: asgi_lifespan package capabilities and testing")

try:
    print("asgi_lifespan provides testing utilities for ASGI applications")
    
    # Package analysis
    attrs = [attr for attr in dir(asgi_lifespan) if not attr.startswith('_')]
    print(f"Available components: {attrs}")
    
    # Demonstrate testing utility usage
    print("\nTesting utility demonstration:")
    
    # Create a test ASGI app for demonstration
    test_resources = {"startup_called": False, "shutdown_called": False}
    
    async def test_asgi_app(scope, receive, send):
        if scope["type"] == "lifespan":
            # ASGI lifespan: keep receiving until shutdown (one receive() then return left LifespanManager waiting 5 s -> TimeoutError)
            while True:
                message = await receive()
            
                if message["type"] == "lifespan.startup":
                    print("Test app: Startup event received")
                    test_resources["startup_called"] = True
                    await send({"type": "lifespan.startup.complete"})
                
                elif message["type"] == "lifespan.shutdown":
                    print("Test app: Shutdown event received") 
                    test_resources["shutdown_called"] = True
                    await send({"type": "lifespan.shutdown.complete"})
                    return
    
    # Test application lifecycle
    print("Testing complete application lifecycle:")
    
    async def run_lifecycle_test():
        async with LifespanManager(test_asgi_app):
            print("App running - startup completed")
            assert test_resources["startup_called"], "Startup not called"
            print("✓ Startup event verified")
            
            # Simulate app usage
            await asyncio.sleep(0.01)
            print("✓ App usage simulated")
        
        print("App stopped - shutdown completed")
        assert test_resources["shutdown_called"], "Shutdown not called"
        print("✓ Shutdown event verified")
        
        return True
    
    # Run the test
    test_passed = await run_lifecycle_test()
    print(f"Lifecycle test passed: {test_passed}")
    
    # Show key capabilities
    print("\nKey asgi_lifespan capabilities:")
    capabilities = [
        "🔄 ASGI lifespan protocol implementation",
        "🏗️ Startup and shutdown event management", 
        "🧪 Testing utilities for ASGI applications",
        "🔗 Context manager for application lifecycle",
        "⚡ Async-first design for modern Python apps",
        "🛡️ Error handling for lifespan events",
        "📦 Integration with ASGI frameworks (FastAPI, Starlette)",
        "🔧 Development and testing workflow support"
    ]
    
    for capability in capabilities:
        print(f"  {capability}")
    
    # Use cases
    print(f"\nCommon use cases:")
    use_cases = [
        "Testing ASGI applications in unit tests",
        "Managing database connections in web apps",
        "Initializing caches and external services",
        "Setting up background tasks and schedulers",
        "Resource cleanup on application shutdown",
        "Development and debugging of ASGI apps",
        "Integration testing of microservices",
        "Health check and monitoring setup"
    ]
    
    for use_case in use_cases:
        print(f"• {use_case}")
    
    # Integration examples
    print(f"\nFramework integration examples:")
    integrations = [
        "FastAPI: Testing startup/shutdown events",
        "Starlette: Managing application state",
        "Quart: Async web application testing",
        "Custom ASGI: Implementing lifespan protocol",
        "Testing: Unit test setup and teardown",
        "CI/CD: Automated application lifecycle testing"
    ]
    
    for integration in integrations:
        print(f"• {integration}")
    
    print("\nasgi_lifespan simplifies ASGI application lifecycle management")
    print("asgi_lifespan example 5 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    import traceback
    traceback.print_exc()
    raise
    
print("Example 5 finished")